In [ ]:
# test용 코드들 임
# SVC_HyperOpt
from hyperopt import fmin, tpe, STATUS_OK, Trials
from sklearn.model_selection import cross_val_score
import numpy as np

# Search Space 정의
lsvc_search_space = {
    'C': hp.loguniform('C', np.log(0.001), np.log(1000)),  # 정규화 강도 (작을수록 강한 정규화)
    'class_weight': hp.choice('class_weight', [None, 'balanced']),  # 클래스 가중치
    'max_iter': hp.quniform('max_iter', 1000, 10000, 1000),  # 최대 반복 횟수
    'tol': hp.loguniform('tol', np.log(1e-5), np.log(1e-2)),  # 수렴 허용 오차
    'dual': hp.choice('dual', [False, True]),  # dual formulation (n_samples > n_features일 때 False 권장)
}

# Objective 함수
def objective(params):
    # 파라미터 타입 변환
    params['C'] = float(params['C'])
    params['max_iter'] = int(params['max_iter'])
    params['tol'] = float(params['tol'])
    
    # 모델 생성
    model = LinearSVC(
        C=params['C'],
        class_weight=params['class_weight'],
        max_iter=params['max_iter'],
        tol=params['tol'],
        dual=params['dual'],
        random_state=team_rs
    )
    
    # 교차 검증
    try:
        scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')  # 또는 'f1'
        score = scores.mean()
    except Exception as e:
        print(f"Error: {e}")
        return {'loss': 1.0, 'status': STATUS_OK}
    
    # HyperOpt는 최소화하므로 음수로 반환
    return {'loss': -score, 'status': STATUS_OK}

# 최적화 실행
trials = Trials()
best_params = fmin(
    fn        = objective,
    space     = lsvc_search_space,
    algo      = tpe.suggest,
    max_evals = 50,  # 시도 횟수
    trials    = trials,
    rstate    = np.random.default_rng(42)
)

print("Best parameters:", best_params)

# 최적 파라미터로 최종 모델 학습
# best_params에서 파라미터 추출 및 변환
final_params = {
    'C': float(best_params['C']),
    'class_weight': [None, 'balanced'][best_params['class_weight']],  # choice는 인덱스로 반환됨
    'max_iter': int(best_params['max_iter']),
    'tol': float(best_params['tol']),
    'dual': [False, True][best_params['dual']],  # choice는 인덱스로 반환됨
}

# 최종 모델 학습
final_model = LinearSVC(**final_params, random_state=42)
final_model.fit(X_train, y_train)

# 예측
y_pred = final_model.predict(X_test)

# 평가
from sklearn.metrics import classification_report, roc_auc_score
print(classification_report(y_test, y_pred))

In [ ]:
import time
import pickle
import os
from hyperopt import fmin, tpe, STATUS_OK, Trials, hp
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score, confusion_matrix
import numpy as np


def hyperopt_tune(model_class, search_space, X_train, y_train, X_test, y_test, 
                  max_evals=100, cv=5, scoring='roc_auc', 
                  random_state=42, save_path=None, verbose=True, use_gpu=False):
    """
    HyperOpt를 사용한 모델 하이퍼파라미터 최적화 함수
    
    Parameters:
    -----------
    model_class : class
        sklearn 모델 클래스 (예: LinearSVC, XGBClassifier)
    search_space : dict
        hyperopt search space 딕셔너리
    X_train : array-like
        학습 데이터
    y_train : array-like
        학습 레이블
    X_test : array-like
        테스트 데이터
    y_test : array-like
        테스트 레이블
    max_evals : int, default=100
        최적화 시도 횟수
    cv : int, default=5
        교차 검증 fold 수
    scoring : str, default='roc_auc'
        평가 지표 ('roc_auc', 'f1', 'recall' 등)
    random_state : int, default=42
        랜덤 시드
    save_path : str or None, default=None
        모델 저장 경로 (None이면 저장 안 함)
    verbose : bool, default=True
        출력 여부 (True: 상세 출력, False: 최소 출력)
    use_gpu : bool, default=False
        GPU 사용 여부 (XGBoost, LightGBM 등에서 지원)
    
    Returns:
    --------
    dict : {
        'model': 최적화된 모델,
        'best_params': 최적 하이퍼파라미터,
        'best_score': 최적 점수,
        'trials': hyperopt trials 객체,
        'metrics': 전체 평가 지표 딕셔너리,
        'confusion_matrix': 혼동 행렬,
        'result_dict': 결과 딕셔너리,
        'elapsed_time': 실행 시간
    }
    
    Examples:
    ---------
    >>> from sklearn.svm import LinearSVC
    >>> from hyperopt import hp
    >>> import numpy as np
    >>> 
    >>> # Search Space 정의
    >>> lsvc_search_space = {
    ...     'C': hp.loguniform('C', np.log(0.001), np.log(1000)),
    ...     'class_weight': hp.choice('class_weight', [None, 'balanced']),
    ...     'max_iter': hp.quniform('max_iter', 1000, 10000, 1000),
    ...     'tol': hp.loguniform('tol', np.log(1e-5), np.log(1e-2)),
    ...     'dual': hp.choice('dual', [False, True]),
    ... }
    >>> 
    >>> # HyperOpt 실행
    >>> result = hyperopt_tune(
    ...     model_class=LinearSVC,
    ...     search_space=lsvc_search_space,
    ...     X_train=X_train,
    ...     y_train=y_train,
    ...     X_test=X_test,
    ...     y_test=y_test,
    ...     max_evals=50,
    ...     cv=5,
    ...     scoring='roc_auc',
    ...     random_state=42,
    ...     save_path='../models/lsvc_best.pkl',
    ...     verbose=True
    ... )
    >>> 
    >>> # XGBoost 예시
    >>> from xgboost import XGBClassifier
    >>> 
    >>> xgb_search_space = {
    ...     'n_estimators': hp.quniform('n_estimators', 100, 1000, 50),
    ...     'max_depth': hp.quniform('max_depth', 3, 10, 1),
    ...     'learning_rate': hp.loguniform('learning_rate', np.log(0.001), np.log(0.3)),
    ...     'subsample': hp.quniform('subsample', 0.5, 1.0, 0.1),
    ...     'colsample_bytree': hp.quniform('colsample_bytree', 0.5, 1.0, 0.1),
    ... }
    >>> 
    >>> # XGBoost GPU 사용 예시
    >>> from xgboost import XGBClassifier
    >>> 
    >>> result = hyperopt_tune(
    ...     model_class=XGBClassifier,
    ...     search_space=xgb_search_space,
    ...     X_train=X_train,
    ...     y_train=y_train,
    ...     X_test=X_test,
    ...     y_test=y_test,
    ...     max_evals=100,
    ...     save_path='../models/xgb_best.pkl',
    ...     use_gpu=True  # GPU 사용
    ... )
    >>> 
    >>> # 결과 사용
    >>> best_model = result['model']
    >>> best_params = result['best_params']
    >>> metrics = result['metrics']
    """
    
    # 모델 이름 자동 추출
    model_name = model_class.__name__
    
    # GPU 설정 메시지
    if verbose and use_gpu:
        print(f"GPU 모드 활성화 ({model_name})")
    
    if verbose:
        print("=" * 50)
        print(f"  {model_name} 튜닝 시작")
        print("=" * 50)
    
    start_time = time.time()  # 시작 시간 기록
    
    # Objective 함수 정의
    def objective(params):
        # 파라미터 타입 변환 (hyperopt는 float로 반환하므로 int 변환 필요)
        converted_params = {}
        for key, value in params.items():
            # quniform으로 정의된 정수형 파라미터 변환
            if key in ['n_estimators', 'max_depth', 'min_child_weight', 'max_iter', 'scale_pos_weight']:
                converted_params[key] = int(value)
            else:
                converted_params[key] = value
        
        # random_state 추가
        converted_params['random_state'] = random_state
        
        # GPU 설정 추가 (모델별로 다름)
        if use_gpu:
            # XGBoost
            if 'XGB' in model_name:
                converted_params['tree_method'] = 'gpu_hist'
                converted_params['gpu_id'] = 0
            # LightGBM
            elif 'LGBM' in model_name or 'LightGBM' in model_name:
                converted_params['device'] = 'gpu'
                converted_params['gpu_platform_id'] = 0
                converted_params['gpu_device_id'] = 0
            # CatBoost
            elif 'CatBoost' in model_name:
                converted_params['task_type'] = 'GPU'
                converted_params['devices'] = '0'
        
        # 모델 생성
        try:
            model = model_class(**converted_params)
        except Exception as e:
            print(f"모델 생성 오류: {e}")
            return {'loss': 1.0, 'status': STATUS_OK}
        
        # 교차 검증
        try:
            scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=scoring)
            score = scores.mean()
        except Exception as e:
            print(f"교차 검증 오류: {e}")
            return {'loss': 1.0, 'status': STATUS_OK}
        
        # HyperOpt는 최소화하므로 음수로 반환
        return {'loss': -score, 'status': STATUS_OK}
    
    # 최적화 실행
    trials = Trials()
    best_params = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=max_evals,
        trials=trials,
        rstate=np.random.default_rng(random_state)
    )
    
    # 걸린 시간 계산
    elapsed_time = time.time() - start_time
    if verbose:
        print(f"튜닝 시간: {elapsed_time:.2f}초")
    
    # 최적 점수 추출
    best_score = -trials.best_trial['result']['loss']
    if verbose:
        print(f"최적 {scoring}: {best_score:.4f}")
    
    # best_params 변환 (choice 타입 처리)
    final_params = {}
    for key, value in best_params.items():
        # choice 파라미터 처리
        if key in search_space:
            space_def = search_space[key]
            # hp.choice인 경우 원래 값으로 변환
            if hasattr(space_def, 'name') and 'choice' in str(type(space_def)):
                # search_space에서 choice 옵션 추출
                choice_options = space_def.pos_args[0].obj
                final_params[key] = choice_options[int(value)]
            # quniform으로 정의된 정수형 파라미터
            elif key in ['n_estimators', 'max_depth', 'min_child_weight', 'max_iter', 'scale_pos_weight']:
                final_params[key] = int(value)
            else:
                final_params[key] = float(value)
        else:
            final_params[key] = value
    
    # random_state 추가
    final_params['random_state'] = random_state
    
    # GPU 설정 추가 (최종 모델)
    if use_gpu:
        if 'XGB' in model_name:
            final_params['tree_method'] = 'gpu_hist'
            final_params['gpu_id'] = 0
        elif 'LGBM' in model_name or 'LightGBM' in model_name:
            final_params['device'] = 'gpu'
            final_params['gpu_platform_id'] = 0
            final_params['gpu_device_id'] = 0
        elif 'CatBoost' in model_name:
            final_params['task_type'] = 'GPU'
            final_params['devices'] = '0'
    
    # 최종 모델 학습
    final_model = model_class(**final_params)
    final_model.fit(X_train, y_train)
    
    # 예측
    y_pred = final_model.predict(X_test)
    
    # 확률 예측 (가능한 경우)
    try:
        if hasattr(final_model, 'predict_proba'):
            y_proba = final_model.predict_proba(X_test)[:, 1]
        elif hasattr(final_model, 'decision_function'):
            y_proba = final_model.decision_function(X_test)
        else:
            y_proba = None
    except:
        y_proba = None
    
    # 전체 평가 지표 계산
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred)
    }
    
    # AUC 계산 (확률 예측 가능한 경우)
    if y_proba is not None:
        try:
            metrics['roc_auc'] = roc_auc_score(y_test, y_proba)
        except:
            metrics['roc_auc'] = roc_auc_score(y_test, y_pred)
    
    # 혼동 행렬
    cm = confusion_matrix(y_test, y_pred)
    
    # 결과 출력
    if verbose:
        print("최적 모델의 전체 평가 점수:")
        for metric_name, metric_value in metrics.items():
            print(f"- {metric_name}: {metric_value:.4f}")
        
        print("최적 하이퍼파라미터:")
        for param_name, param_value in final_params.items():
            print(f"-{param_name}: {param_value}")
    
    # 모델 저장
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        with open(save_path, 'wb') as f:
            pickle.dump(final_model, f)
        file_size = os.path.getsize(save_path) / (1024 * 1024)  # MB 단위
        if verbose:
            print(f"✓ 모델 저장 완료: {save_path}")
            print(f"  파일 크기: {file_size:.2f} MB")
    
    # 결과 딕셔너리 출력 (사용자가 원하는 형식)
    if verbose:
        print(f"folder = {os.path.dirname(save_path) if save_path else 'Not saved'}")
    
    result_dict = {
        'AUC': round(metrics.get('roc_auc', 0), 4),
        '정확도': round(metrics['accuracy'], 4),
        '정밀도': round(metrics['precision'], 4),
        '재현율': round(metrics['recall'], 4),
        'F1': round(metrics['f1'], 4)
    }
    
    if verbose:
        print(result_dict)
        print({'오차행렬': cm})
        print(f"실행 시간: {elapsed_time}")
        print(f"하이퍼파라미터: {final_params}")
    
    # 반환
    return {
        'model': final_model,
        'best_params': final_params,
        'best_score': best_score,
        'trials': trials,
        'metrics': metrics,
        'confusion_matrix': cm,
        'result_dict': result_dict,
        'elapsed_time': elapsed_time
    }


# ========================================
# 사용 예시
# ========================================

if __name__ == "__main__":
    from sklearn.svm import LinearSVC
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split
    
    # 예시 데이터 생성
    X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, 
                                weights=[0.9, 0.1], random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Search Space 정의
    lsvc_search_space = {
        'C': hp.loguniform('C', np.log(0.001), np.log(1000)),
        'class_weight': hp.choice('class_weight', [None, 'balanced']),
        'max_iter': hp.quniform('max_iter', 1000, 10000, 1000),
        'tol': hp.loguniform('tol', np.log(1e-5), np.log(1e-2)),
        'dual': hp.choice('dual', [False, True]),
    }
    
    # HyperOpt 실행
    result = hyperopt_tune(
        model_class=LinearSVC,
        search_space=lsvc_search_space,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        max_evals=50,
        cv=5,
        scoring='roc_auc',
        random_state=42,
        save_path='../models/lsvc_best.pkl',
        verbose=True
    )
    
    # 결과 사용
    best_model = result['model']
    best_params = result['best_params']
    print("\n최종 결과:", result['result_dict'])